## Libraries import

In [12]:
import os
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import pandas as pd


## Model

In [17]:
class FaceRecognitionModel(nn.Module):
    def __init__(self, num_classes):
        super(FaceRecognitionModel, self).__init__()
        weights = MobileNet_V2_Weights.DEFAULT
        base_model = mobilenet_v2(weights=weights)
        self.features = base_model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.embedding = nn.Linear(base_model.last_channel, 256)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).view(x.size(0), -1)
        embedding = self.embedding(x)
        logits = self.classifier(embedding)
        return logits, embedding

## Loss Function

In [18]:
criterion = nn.CrossEntropyLoss()

## Accuracy Function

In [19]:
def accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    return correct / labels.size(0)

## Train Loop

In [20]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            labels = batch['identity'].to(device)

            logits, _ = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy(logits, labels)

    return total_loss / len(dataloader), total_acc / len(dataloader)

In [21]:
def train_and_validate(model, train_loader, val_loader, optimizer, criterion, device, epochs=10):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_acc = 0.0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]", leave=False)

        for batch in train_bar:
            images = batch['image'].to(device)
            labels = batch['identity'].to(device)

            optimizer.zero_grad()
            logits, _ = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_acc += accuracy(logits, labels)

        avg_train_loss = total_loss / len(train_loader)
        avg_train_acc = total_acc / len(train_loader)

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print("✅ Saved new best model")

    print(f"📈 Best validation accuracy: {best_val_acc:.4f}")

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceRecognitionModel(num_classes=dataset.num_identities).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

train_and_validate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

Epoch 1/10 | Train Loss: 8.6056, Acc: 0.0023 | Val Loss: 12.9125, Acc: 0.0000


Epoch 2/10 | Train Loss: 6.9971, Acc: 0.0226 | Val Loss: 15.2752, Acc: 0.0000


KeyboardInterrupt: 